## RAG pipe - Data ingestion to vector DB pipeline

In [65]:
import  os
from multiprocessing import context

from langchain_classic import text_splitter
from  langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from  langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Read all the pdf's inside the directory


In [66]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: 14. Elevator System.pdf
  ✓ Loaded 7 pages

Processing: Does ChatGPT Know or Does It Guess?.pdf
  ✓ Loaded 4 pages

Processing: 7. Chess Game.pdf
  ✓ Loaded 10 pages

Total documents loaded: 21


In [67]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2025-02-19T04:12:43+05:30', 'author': 'Ashish Misal', 'keywords': 'System Design Interview Questions', 'moddate': '2025-02-19T04:12:43+05:30', 'source': '../data/pdf_files/14. Elevator System.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': '14. Elevator System.pdf', 'file_type': 'pdf'}, page_content='1 \n \n14. Elevator System \nHere’s a comprehensive elevator system design following the structured approach \nyou requested:   \nStep 1: Outline Use Cases and Constraints   \nUse Cases (In-Scope) \n1. Call Elevator from a Floor: A user presses an up or down button to call the \nelevator.   \n2. Select Destination Floor: Once inside, a user selects a destination floor.   \n3. Elevator Movement: The elevator moves to the requested floors in an \noptimized manner.   \n4. Door Operation: The doors open and close at the requested floors.   \n5. Emergenc

## Text Splitting get into chunks


In [68]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [69]:
chunks=split_documents(all_pdf_documents)
chunks

Split 21 documents into 27 chunks

Example chunk:
Content: 1 
 
14. Elevator System 
Here’s a comprehensive elevator system design following the structured approach 
you requested:   
Step 1: Outline Use Cases and Constraints   
Use Cases (In-Scope) 
1. Call ...
Metadata: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2025-02-19T04:12:43+05:30', 'author': 'Ashish Misal', 'keywords': 'System Design Interview Questions', 'moddate': '2025-02-19T04:12:43+05:30', 'source': '../data/pdf_files/14. Elevator System.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': '14. Elevator System.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2025-02-19T04:12:43+05:30', 'author': 'Ashish Misal', 'keywords': 'System Design Interview Questions', 'moddate': '2025-02-19T04:12:43+05:30', 'source': '../data/pdf_files/14. Elevator System.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': '14. Elevator System.pdf', 'file_type': 'pdf'}, page_content='1 \n \n14. Elevator System \nHere’s a comprehensive elevator system design following the structured approach \nyou requested:   \nStep 1: Outline Use Cases and Constraints   \nUse Cases (In-Scope) \n1. Call Elevator from a Floor: A user presses an up or down button to call the \nelevator.   \n2. Select Destination Floor: Once inside, a user selects a destination floor.   \n3. Elevator Movement: The elevator moves to the requested floors in an \noptimized manner.   \n4. Door Operation: The doors open and close at the requested floors.   \n5. Emergenc

### Embedding and vectorStoreDB

In [70]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

### EmbeddingManager

In [71]:
class EmbeddingManager:
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        Intialise the embedding manager

        Args
            model_name: Huggingface model name for sentece embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """ Load the sentence transformation model """
        try:
            print(f"Loadig embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model load successfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model : {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


embedding_manager=EmbeddingManager()
embedding_manager

Loadig embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5310.42it/s]


Model load successfully. Embedding dimention: 384


/var/folders/yb/z0d63kfs0mqg4sq6dtzf7z7c0000gn/T/ipykernel_4168/1365266590.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model load successfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [72]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 26


### Convert the text to embeddings

In [73]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

Generating embeddings for 27 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]

Generated embeddings with shape: (27, 384)


array([[ 0.00251973, -0.0144775 ,  0.00602861, ...,  0.08282335,
        -0.0882396 , -0.03104062],
       [ 0.00347713, -0.02617292,  0.00036656, ...,  0.03785408,
        -0.00258154, -0.02606235],
       [ 0.00307126, -0.06992473,  0.00560767, ...,  0.09747078,
        -0.06339724, -0.00459053],
       ...,
       [ 0.10117126, -0.0197975 , -0.0676662 , ..., -0.03631444,
        -0.02481047,  0.01507055],
       [ 0.09626116,  0.04643444, -0.05005373, ...,  0.03817819,
        -0.0488732 , -0.01036762],
       [ 0.03900503,  0.06581642, -0.06495894, ..., -0.01485715,
        -0.04289125,  0.02473497]], shape=(27, 384), dtype=float32)

### Store embeddings into VectorStore

In [74]:
vectorstore.add_documents(chunks, embeddings)

Adding 27 documents to vector store...
Successfully added 27 documents to vector store
Total documents in collection: 53


# Retriever pipeline from VectorStore

In [75]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [76]:
rag_retriever.retrieve("Base Model vs. AI Assistant")

Retrieving documents for query: 'Base Model vs. AI Assistant'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

# Integration VectorDB context pipeline with LLM output

### Simple RAG pipeline with Groq LLM

In [81]:
from langchain_groq import ChatGroq
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
### Initialise the GROQ LLM
llm=ChatGroq(api_key=groq_api_key, model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

### Simple RAG function (Retrieve context + generate response)
def rag_simple(query:str, retriever:RAGRetriever, llm:ChatGroq , top_k=3):
    ## Retrieve the context
    results=retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    ## generate the answer using GROK_LLM
    prompt=f"""
            Use the following context to answer the question concisely.
            Context : {context}
            Question : {query}
            """
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

answer=rag_simple("What is redis ? ", rag_retriever, llm)
answer=rag_simple("Who is Rajesh? ", rag_retriever, llm)
answer=rag_simple("What are core components in designing chess game ?  ", rag_retriever, llm)
answer


Retrieving documents for query: 'What is redis ? '
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Retrieving documents for query: 'Who is Rajesh? '
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.76it/s]


Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Retrieving documents for query: 'What are core components in designing chess game ?  '
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'**Core components of a chess‑game design**\n\n| Layer | Key components |\n|-------|----------------|\n| **Domain / Engine** | • Board representation (8×8 grid)  <br>• Piece hierarchy (Pawn, Rook, Knight, Bishop, Queen, King)  <br>• Move generation & validation (legal moves, check, check‑mate, stalemate)  <br>• Game state machine (turn, castling rights, en‑passant, promotion) |\n| **Persistence** | • Database schema for active/completed games (game ID, players, board snapshot, move history)  <br>• Move log for replay/review |\n| **Networking** | • Client‑server architecture (stateless or stateful server)  <br>• Low‑latency communication (WebSocket or TCP)  <br>• Synchronization of game state between peers |\n| **User Interface** | • Client‑side UI (board rendering, piece drag‑drop or click)  <br>• Move history view, timers, chat (optional) |\n| **Application Layer** | • Service layer that receives client moves, validates via engine, updates DB, broadcasts to opponent  <br>• Error handl

# Enhanced RAG Pipeline features

In [88]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query:str, retriever: RAGRetriever, llm, top_k=5, min_score=0, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # results=retriever.retrieve(query, top_k=top_k)
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Board Class in designing chess game ? ", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("\nSources:", result['sources'])
print("\nConfidence:", result['confidence'])
print("\nContext Preview:", result['context'][:300])

Retrieving documents for query: 'Board Class in designing chess game ? '
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: **Board Class – Core responsibilities**

| Responsibility | What it does |
|----------------|--------------|
| **Represent the 8×8 grid** | Holds an 8‑row × 8‑column array of `Box` objects. Each `Box` may contain a `Piece` or be empty. |
| **Initialize the board** | In the constructor or an `init()` method, place the standard starting pieces on the correct boxes. |
| **Provide access to boxes** | `Box getBox(int row, int col)` or `Box getBox(Position pos)` for other classes (Game, Move) to query or update. |
| **Move pieces** | `boolean movePiece(Position from, Position to)` – updates the source and destination boxes, handles captures, and returns whether the move was legal (delegated to `Piece` logic). |
| **Query state** | `boolean isOccupied(Position pos)`, `Piece getPiece(Position pos)`, `boolean isInCheck(Color color)` (by scanning opponent moves). |
| **Utility helpers** | `List<Piece> getAl